# Part B – Text Representation and Spam Classification

## Objective

This part studies the numerical representation of email data and compares two text representations:

1. Count / word-frequency representation
2. TF-IDF representation

Traditional machine-learning classifiers are trained on the representations and evaluated using accuracy, precision, recall, F1-score, confusion matrix, training time and prediction time.

**Important dataset note:** the supplied `emails.csv` contains `Email No.`, 3000 word-frequency columns and `Prediction`. It therefore already contains a numerical word-frequency representation rather than a raw email-text column. Because of this, `CountVectorizer` cannot correctly be applied to the 3000 numeric columns. The notebook uses the existing word-frequency matrix as the Count-style representation and applies `TfidfTransformer` to that matrix to create the TF-IDF representation.

This avoids inventing or reconstructing raw email text that is not present in the supplied CSV.

## 1. Import Libraries

In [1]:
import pandas as pd
import time

from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfTransformer
from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
    classification_report
)

## 2. Load the Email Dataset

The dataset contains 5,172 emails.

The `Prediction` column is the target:

- `0` = non-spam
- `1` = spam

`Email No.` is an identifier and should not be used as a predictive feature.

In [2]:
emails_df = pd.read_csv("emails.csv")

print("Dataset shape:", emails_df.shape)
display(emails_df.head())
print("Column names:")
print(emails_df.columns[:10].tolist())

Dataset shape: (5172, 3002)


,Email No.,the,to,ect,and,for,of,a,you,hou,...,connevey,jay,valued,lay,infrastructure,military,allowing,ff,dry,Prediction
0,Email 1,0,0,1,0,0,0,2,0,0,...,0,0,0,0,0,0,0,0,0,0
1,Email 2,8,13,24,6,6,2,102,1,27,...,0,0,0,0,0,0,0,1,0,0
2,Email 3,0,0,1,0,0,0,8,0,0,...,0,0,0,0,0,0,0,0,0,0
3,Email 4,0,5,22,0,5,1,51,2,10,...,0,0,0,0,0,0,0,0,0,0
4,Email 5,7,6,17,1,5,2,57,0,9,...,0,0,0,0,0,0,0,1,0,0


Column names:
['Email No.', 'the', 'to', 'ect', 'and', 'for', 'of', 'a', 'you', 'hou']


## 3. Inspect Spam / Non-Spam Distribution

Checking the target distribution helps identify whether the dataset is balanced or imbalanced.

The model evaluation should therefore include precision, recall and F1-score in addition to accuracy.

In [3]:
print("Prediction distribution:")
print(emails_df["Prediction"].value_counts())

print("\nClass proportions:")
print(emails_df["Prediction"].value_counts(normalize=True))

Prediction distribution:
Prediction
0    3672
1    1500
Name: count, dtype: int64

Class proportions:
Prediction
0    0.709977
1    0.290023
Name: proportion, dtype: float64


## 4. Prepare the Existing Word-Frequency Representation

The CSV contains 3000 word-frequency columns.

Therefore:

- `Email No.` is removed because it is only an identifier.
- `Prediction` is removed from the feature matrix.
- The remaining 3000 columns are already numerical word-frequency features.

This matrix is treated as the **Count/word-frequency representation**.

In [4]:
X_count = emails_df.drop(columns=["Email No.", "Prediction"])
y = emails_df["Prediction"]

print("Count/word-frequency feature shape:", X_count.shape)
print("Target shape:", y.shape)

Count/word-frequency feature shape: (5172, 3000)
Target shape: (5172,)


## 5. Create the TF-IDF Representation

TF-IDF stands for **Term Frequency – Inverse Document Frequency**.

It reduces the relative importance of words that occur frequently across many documents and gives more weight to terms that are more informative.

Because the dataset already contains word-frequency counts, `TfidfTransformer` is the appropriate sklearn transformation here.

This is different from `TfidfVectorizer`, which expects raw text strings.

In [5]:
tfidf_transformer = TfidfTransformer()

X_tfidf = tfidf_transformer.fit_transform(X_count)

print("TF-IDF feature shape:", X_tfidf.shape)

TF-IDF feature shape: (5172, 3000)


## 6. Compare the Number of Features

Both representations contain the same 3000 vocabulary columns because TF-IDF changes the values assigned to the existing features rather than creating a new vocabulary.

Therefore, the representation comparison focuses on **feature values and predictive performance**, while the feature dimensionality remains the same.

In [6]:
representation_summary = pd.DataFrame({
    "Representation": ["Count / Word Frequency", "TF-IDF"],
    "Number of Features": [X_count.shape[1], X_tfidf.shape[1]],
    "Number of Samples": [X_count.shape[0], X_tfidf.shape[0]]
})

display(representation_summary)

,Representation,Number of Features,Number of Samples
0,Count / Word Frequency,3000,5172
1,TF-IDF,3000,5172


## 7. Train-Test Split

The same 80:20 stratified split is used for both representations.

Using the same random state and stratification makes the comparison fair because both models are evaluated on the same type of train/test division.

In [7]:
X_count_train, X_count_test, y_train, y_test = train_test_split(
    X_count,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

X_tfidf_train, X_tfidf_test, _, _ = train_test_split(
    X_tfidf,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

print("Count training shape:", X_count_train.shape)
print("Count testing shape:", X_count_test.shape)
print("TF-IDF training shape:", X_tfidf_train.shape)
print("TF-IDF testing shape:", X_tfidf_test.shape)

Count training shape: (4137, 3000)
Count testing shape: (1035, 3000)
TF-IDF training shape: (4137, 3000)
TF-IDF testing shape: (1035, 3000)


## 8. Classification Models

Two traditional text-classification models are used:

### Logistic Regression
Logistic Regression is a strong baseline for high-dimensional sparse text features and provides a simple linear decision boundary.

### Linear SVM
Linear Support Vector Machine is commonly suitable for high-dimensional text representations because it learns a linear boundary efficiently in feature spaces with many dimensions.

The same classifiers are applied to both Count and TF-IDF representations so that the effect of the representation can be compared.

In [8]:
models = {
    "Logistic Regression": LogisticRegression(max_iter=2000),
    "Linear SVM": LinearSVC()
}

## 9. Evaluation Function

For each model and representation, the following are measured:

- Accuracy
- Precision
- Recall
- F1-score
- Training time
- Prediction time
- Confusion matrix

The timing is measured separately for training and prediction.

In [9]:
def evaluate_model(model, X_train, X_test, y_train, y_test):
    start_train = time.time()
    model.fit(X_train, y_train)
    train_time = time.time() - start_train

    start_prediction = time.time()
    y_pred = model.predict(X_test)
    prediction_time = time.time() - start_prediction

    result = {
        "Accuracy": accuracy_score(y_test, y_pred),
        "Precision": precision_score(y_test, y_pred, zero_division=0),
        "Recall": recall_score(y_test, y_pred, zero_division=0),
        "F1-score": f1_score(y_test, y_pred, zero_division=0),
        "Training Time": train_time,
        "Prediction Time": prediction_time,
        "Confusion Matrix": confusion_matrix(y_test, y_pred)
    }

    return result

## 10. Train and Evaluate All Model/Representation Combinations

Four experiments are performed:

1. Count + Logistic Regression
2. Count + Linear SVM
3. TF-IDF + Logistic Regression
4. TF-IDF + Linear SVM

Keeping the models fixed while changing the representation helps isolate the effect of Count versus TF-IDF.

In [10]:
results = []
confusion_matrices = {}

for model_name, model in models.items():

    # Count representation
    count_model = model.__class__(**model.get_params())

    count_result = evaluate_model(
        count_model,
        X_count_train,
        X_count_test,
        y_train,
        y_test
    )

    results.append({
        "Representation": "Count / Word Frequency",
        "Model": model_name,
        "Features": X_count_train.shape[1],
        "Accuracy": count_result["Accuracy"],
        "Precision": count_result["Precision"],
        "Recall": count_result["Recall"],
        "F1-score": count_result["F1-score"],
        "Training Time": count_result["Training Time"],
        "Prediction Time": count_result["Prediction Time"]
    })

    confusion_matrices[("Count / Word Frequency", model_name)] = count_result["Confusion Matrix"]

    # TF-IDF representation
    tfidf_model = model.__class__(**model.get_params())

    tfidf_result = evaluate_model(
        tfidf_model,
        X_tfidf_train,
        X_tfidf_test,
        y_train,
        y_test
    )

    results.append({
        "Representation": "TF-IDF",
        "Model": model_name,
        "Features": X_tfidf_train.shape[1],
        "Accuracy": tfidf_result["Accuracy"],
        "Precision": tfidf_result["Precision"],
        "Recall": tfidf_result["Recall"],
        "F1-score": tfidf_result["F1-score"],
        "Training Time": tfidf_result["Training Time"],
        "Prediction Time": tfidf_result["Prediction Time"]
    })

    confusion_matrices[("TF-IDF", model_name)] = tfidf_result["Confusion Matrix"]

results_df = pd.DataFrame(results)
display(results_df)

C:\Users\H P\AppData\Roaming\Python\Python313\site-packages\sklearn\svm\_base.py:1250: ConvergenceWarning: Liblinear failed to converge, increase the number of iterations.
  warnings.warn(


,Representation,Model,Features,Accuracy,Precision,Recall,F1-score,Training Time,Prediction Time
0,Count / Word Frequency,Logistic Regression,3000,0.982609,0.957792,0.983333,0.970395,15.745191,0.053893
1,TF-IDF,Logistic Regression,3000,0.950725,0.930796,0.896667,0.913413,0.323493,0.000766
2,Count / Word Frequency,Linear SVM,3000,0.979710,0.960396,0.970000,0.965174,3.211089,0.042797
3,TF-IDF,Linear SVM,3000,0.984541,0.964052,0.983333,0.973597,0.097062,0.000691


## 11. Confusion Matrices

The confusion matrices show the number of correct and incorrect spam/non-spam predictions for every experiment.

In [11]:
for key, cm in confusion_matrices.items():
    representation, model_name = key

    print("=" * 60)
    print(representation, "-", model_name)
    print("=" * 60)
    print(cm)
    print()

Count / Word Frequency - Logistic Regression
[[722  13]
 [  5 295]]

TF-IDF - Logistic Regression
[[715  20]
 [ 31 269]]

Count / Word Frequency - Linear SVM
[[723  12]
 [  9 291]]

TF-IDF - Linear SVM
[[724  11]
 [  5 295]]



## 12. Detailed Classification Reports

The classification reports show precision, recall and F1-score separately for spam and non-spam classes.

This is useful because a spam classifier should be examined not only by overall accuracy but also by how well it identifies spam and avoids incorrectly labelling legitimate emails as spam.

In [12]:
for representation, X_train_rep, X_test_rep in [
    ("Count / Word Frequency", X_count_train, X_count_test),
    ("TF-IDF", X_tfidf_train, X_tfidf_test)
]:
    for model_name, model in models.items():

        fresh_model = model.__class__(**model.get_params())
        fresh_model.fit(X_train_rep, y_train)
        y_pred = fresh_model.predict(X_test_rep)

        print("=" * 70)
        print(representation, "-", model_name)
        print("=" * 70)

        print(classification_report(
            y_test,
            y_pred,
            target_names=["Non-Spam", "Spam"],
            zero_division=0
        ))

Count / Word Frequency - Logistic Regression
              precision    recall  f1-score   support

    Non-Spam       0.99      0.98      0.99       735
        Spam       0.96      0.98      0.97       300

    accuracy                           0.98      1035
   macro avg       0.98      0.98      0.98      1035
weighted avg       0.98      0.98      0.98      1035



C:\Users\H P\AppData\Roaming\Python\Python313\site-packages\sklearn\svm\_base.py:1250: ConvergenceWarning: Liblinear failed to converge, increase the number of iterations.
  warnings.warn(


Count / Word Frequency - Linear SVM
              precision    recall  f1-score   support

    Non-Spam       0.99      0.98      0.99       735
        Spam       0.96      0.97      0.97       300

    accuracy                           0.98      1035
   macro avg       0.97      0.98      0.98      1035
weighted avg       0.98      0.98      0.98      1035

TF-IDF - Logistic Regression
              precision    recall  f1-score   support

    Non-Spam       0.96      0.97      0.97       735
        Spam       0.93      0.90      0.91       300

    accuracy                           0.95      1035
   macro avg       0.94      0.93      0.94      1035
weighted avg       0.95      0.95      0.95      1035

TF-IDF - Linear SVM
              precision    recall  f1-score   support

    Non-Spam       0.99      0.99      0.99       735
        Spam       0.96      0.98      0.97       300

    accuracy                           0.98      1035
   macro avg       0.98      0.98      0.98

## 13. Compare Count and TF-IDF

The final table allows comparison of:

- predictive performance,
- feature dimensionality,
- training time,
- prediction time.

Since the supplied CSV already contains 3000 word-frequency columns, both representations have the same number of features. The main difference is the numerical weighting of those features.

In [13]:
display(
    results_df.sort_values(
        by=["Representation", "Model"]
    ).reset_index(drop=True)
)

,Representation,Model,Features,Accuracy,Precision,Recall,F1-score,Training Time,Prediction Time
0,Count / Word Frequency,Linear SVM,3000,0.979710,0.960396,0.970000,0.965174,3.211089,0.042797
1,Count / Word Frequency,Logistic Regression,3000,0.982609,0.957792,0.983333,0.970395,15.745191,0.053893
2,TF-IDF,Linear SVM,3000,0.984541,0.964052,0.983333,0.973597,0.097062,0.000691
3,TF-IDF,Logistic Regression,3000,0.950725,0.930796,0.896667,0.913413,0.323493,0.000766


## 14. Save the Results

The comparison table is saved for use in the project report and GitHub repository.

In [14]:
results_df.to_csv("text_classification_results.csv", index=False)
print("Saved: text_classification_results.csv")

Saved: text_classification_results.csv


## 15. Part B Conclusion

The email data was evaluated using its existing word-frequency representation and a TF-IDF transformation.

The two representations were tested with Logistic Regression and Linear SVM using the same train-test split.

The actual experiment results should be discussed using the values in `results_df`. The comparison should mention predictive performance as well as training and prediction time.

### Dataset limitation

The supplied `emails.csv` does not contain a raw email-text column. Therefore, `CountVectorizer` and `TfidfVectorizer` cannot be honestly applied directly to this file. If the instructor specifically requires those two vectorizer classes, the project needs the original raw-email text dataset. With the current CSV, `TfidfTransformer` is the technically appropriate way to convert the existing count matrix into TF-IDF values.